In [1]:
import os
import pandas as pd
from torchvision.io import decode_image
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from torchvision.transforms import ToTensor
from PIL import Image
import torch.nn.functional as F

In [3]:
class ImageDatasetFromFolders(Dataset):
    def __init__(self, path, transform=ToTensor):
        self.path = path
        self.class_names = sorted(os.listdir(self.path))
        self.transform = transform
        complete_dataset = []
        for dir in os.listdir(self.path):
            for image in os.listdir(os.path.join(self.path, dir)):
                complete_dataset.append((dir, os.path.join(self.path, dir, image)))
        self.complete_dataset = complete_dataset
        
    def __len__(self):
        return len(self.complete_dataset)
    
    def get_example(self, idx):
        img_path = self.complete_dataset[idx][1]
        image = Image.open(img_path)
        label = self.complete_dataset[idx][0]
        label = self.class_names.index(label)
        if self.transform:
            image = self.transform(image)
        return image, label
    
    def scale_image(self, img):
        max_pixel = np.max(np.array(img))
        img = img/max_pixel
        return img
    
    def __getitem__(self, idx):
        img_path = self.complete_dataset[idx][1]
        image = Image.open(img_path)
        label = self.complete_dataset[idx][0]
#         print(label)
        label = self.class_names.index(label)
#         print(label)
        if self.transform:
            image = self.transform(image)
#            image = self.scale_image(image)
        return image, label

In [4]:
base = os.getcwd()
train_dir = os.path.join(base,'Train')
test_dir = os.path.join(base,'Test')
val_dir = os.path.join(base,'Validation')

image_size = (256,256)
transform = transforms.Compose([transforms.Resize(image_size),transforms.ToTensor()])
#transform = transforms.Compose([transforms.PILToTensor(), transforms.Resize(image_size)])
# transform = transforms.Compose([transforms.Resize(image_size)])
train_data = ImageDatasetFromFolders(train_dir, transform=transform)
test_data = ImageDatasetFromFolders(test_dir, transform=transform)
val_data = ImageDatasetFromFolders(val_dir, transform=transform)
print((train_data[2][0]).dtype) #check the type of the data to make sure it is torch.float32
print(train_data[1000][0]) #check if the data has been scaled
print(test_data.__len__())
print(val_data.__len__())

torch.float32
tensor([[[0.5490, 0.5608, 0.5804,  ..., 0.9961, 0.9961, 1.0000],
         [0.5451, 0.5529, 0.5647,  ..., 1.0000, 1.0000, 1.0000],
         [0.5451, 0.5529, 0.5647,  ..., 1.0000, 1.0000, 0.9961],
         ...,
         [0.5686, 0.5294, 0.4902,  ..., 0.4000, 0.3961, 0.3804],
         [0.5373, 0.4980, 0.4549,  ..., 0.4000, 0.3961, 0.3804],
         [0.5176, 0.4941, 0.4471,  ..., 0.4039, 0.3922, 0.3765]],

        [[0.6588, 0.6706, 0.6824,  ..., 1.0000, 1.0000, 0.9961],
         [0.6588, 0.6667, 0.6745,  ..., 1.0000, 0.9961, 0.9961],
         [0.6549, 0.6627, 0.6706,  ..., 1.0000, 1.0000, 0.9961],
         ...,
         [0.4392, 0.3765, 0.3176,  ..., 0.5686, 0.5647, 0.5608],
         [0.4157, 0.3569, 0.2980,  ..., 0.5686, 0.5647, 0.5608],
         [0.4000, 0.3608, 0.2980,  ..., 0.5725, 0.5647, 0.5569]],

        [[0.5020, 0.5137, 0.5294,  ..., 0.9569, 0.9843, 0.9804],
         [0.5020, 0.5098, 0.5176,  ..., 0.9922, 0.9961, 0.9882],
         [0.4980, 0.5059, 0.5137,  ..., 1.00